# Cross-sectional price momentum

A textbook 12-1 momentum ranking on a universe of assets: skip the most recent month, rank by the prior eleven months, hold the top quintile for a month. The point is the mechanics (no look-ahead, monthly rebalancing, turnover), not a performance claim. Data is synthetic unless `USE_YFINANCE=1`.

In [ ]:
import os
from pathlib import Path

# All inputs and outputs live under DATA_DIR (relative to the repo by default).
DATA_DIR = Path(os.environ.get("DATA_DIR", "data"))
OUT_DIR = DATA_DIR / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

USE_YFINANCE = os.environ.get("USE_YFINANCE", "0") == "1"
TICKERS = ["XLB", "XLE", "XLF", "XLI", "XLK", "XLP", "XLU", "XLV", "XLY", "GLD", "TLT", "EFA", "EEM", "IWM", "QQQ"]
LOOKBACK, SKIP, TOP_N = 12, 1, 3


def load_monthly():
    if USE_YFINANCE:
        import yfinance as yf
        px = yf.download(TICKERS, start="2010-01-01", auto_adjust=True, progress=False)["Close"]
    else:
        rng = np.random.default_rng(SEED)
        idx = pd.bdate_range("2010-01-01", periods=252 * 12)
        drift = rng.normal(0.0002, 0.0002, len(TICKERS))
        px = pd.DataFrame(100 * np.exp(np.cumsum(rng.normal(drift, 0.012, (len(idx), len(TICKERS))), axis=0)),
                          index=idx, columns=TICKERS)
    return px.resample("ME").last()


monthly = load_monthly()
mret = monthly.pct_change()

In [ ]:
# Signal at month-end t uses prices up to t - SKIP only; position is held over month t+1.
signal = monthly.shift(SKIP) / monthly.shift(LOOKBACK) - 1
ranks = signal.rank(axis=1, ascending=False)
weights = (ranks <= TOP_N).astype(float)
weights = weights.div(weights.sum(axis=1), axis=0).fillna(0)

strategy = (weights.shift(1) * mret).sum(axis=1).loc[signal.dropna(how="all").index[1]:]
benchmark = mret.mean(axis=1).loc[strategy.index]
turnover = weights.diff().abs().sum(axis=1).loc[strategy.index] / 2

pd.DataFrame({"momentum_top_n": (1 + strategy).cumprod(), "equal_weight": (1 + benchmark).cumprod()}).plot(
    figsize=(11, 4), title="Growth of 1 (synthetic unless USE_YFINANCE=1)")
plt.show()
print(f"average monthly one-way turnover: {turnover.mean():.1%}")